In [17]:
import pandas as pd

In [ ]:
df_funding = pd.read_csv("../../data/fts_requirements_funding_global.csv")
df_funding_cluster = pd.read_csv("../../data/fts_requirements_funding_globalcluster_global.csv")

df_funding['year'] = df_funding['year'].astype(str)
df_funding_cluster['year'] = df_funding_cluster['year'].astype(str)

In [19]:
df_funding_cluster

,countryCode,id,name,code,startDate,endDate,year,clusterCode,cluster,requirements,funding,percentFunded
0,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,26480.0,Coordination and support services,24700000.0,1805186.0,7.0
1,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,3.0,Education,60040000.0,18138612.0,30.0
2,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,4.0,Emergency Shelter and NFI,160285975.0,5814964.0,4.0
3,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,6.0,Food Security,651101609.0,50221603.0,8.0
4,AFG,1502,Afghanistan Humanitarian Needs and Response Pl...,HAFG26,2026-01-01,2026-12-31,2026,7.0,Health,190768456.0,30587554.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...
10625,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,26479.0,Multi-sector,557000.0,2244422.0,403.0
10626,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,10.0,Protection,1500000.0,NaN,NaN
10627,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,11.0,Water Sanitation Hygiene,1500000.0,300035.0,20.0
10628,ZWE,98,Humanitarian Crisis in Southern Africa 2002 - ...,CZWE0203,2002-07-01,2003-06-30,2002,NaN,Not specified,NaN,NaN,NaN


In [36]:
merge_keys = ['id', 'code', 'countryCode', 'year', 'name']

df_merged = pd.merge(
    df_funding, 
    df_funding_cluster, 
    on=merge_keys, 
    how='inner',
    suffixes=('_plan_total', '_cluster_specific')
)

numeric_cols = [
    'requirements_plan_total', 'funding_plan_total', 
    'requirements_cluster_specific', 'funding_cluster_specific'
]
for col in numeric_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')


In [35]:
import pandas as pd

def aggregate_needs(year):
    pd.options.display.float_format = '{:,.0f}'.format
    df_needs = pd.read_csv(f"../../data/hpc_hno_{year}.csv", low_memory=False)
    df_needs = df_needs[1:] 

    df_needs["In Need"] = df_needs["In Need"].astype(str).str.replace(',', '')
    df_needs["In Need"] = pd.to_numeric(df_needs["In Need"], errors='coerce').fillna(0)

    df_clean = df_needs[df_needs['Admin 1 PCode'].isna() & df_needs['Category'].isna()]

    df_pivot = pd.pivot_table(
        df_clean, 
        values='In Need', 
        index='Country ISO3', 
        columns='Cluster', 
        aggfunc='sum', 
        fill_value=0
    ).reset_index()

    cluster_cols = [col for col in df_pivot.columns if col != 'Country ISO3']
    df_pivot = df_pivot.rename(columns={col: f"In Need - {col}" for col in cluster_cols})

    return df_pivot


In [39]:
df_needs = aggregate_needs("2024")

cluster_mapping = {
    'In Need - AGR': 'Agriculture',
    'In Need - CCM': 'Camp Coordination / Management', 
    'In Need - CSS': 'Coordination and support services',
    'In Need - EDU': 'Education',
    'In Need - ERY': 'Early Recovery',
    'In Need - FSC': 'Food Security',
    'In Need - HEA': 'Health',
    'In Need - LOG': 'Logistics',
    'In Need - MS':  'Multi-sector',
    'In Need - NUT': 'Nutrition',
    'In Need - PRO': 'Protection',
    'In Need - PRO-CPN': 'Protection - Child Protection',
    'In Need - PRO-GBV': 'Protection - Gender-Based Violence',
    'In Need - PRO-HLP': 'Protection - Housing, Land and Property',
    'In Need - PRO-MIN': 'Protection - Mine Action',
    'In Need - SHL': 'Emergency Shelter and NFI',
    'In Need - WSH': 'Water Sanitation Hygiene',
    'In Need - ALL': 'Total In Need'
}

df_needs.rename(columns=cluster_mapping, inplace=True)

df_needs = pd.melt(
    df_needs,
    id_vars=['Country ISO3'],
    value_vars=[c for c in df_needs.columns if c != 'Country ISO3'],
    var_name='HNO_Cluster_Code',
    value_name='People_In_Need'
)
df_needs["year"] = 2024
df_needs

,Country ISO3,HNO_Cluster_Code,People_In_Need,year
0,AFG,Agriculture,0,2024
1,BFA,Agriculture,0,2024
2,CAF,Agriculture,0,2024
3,CMR,Agriculture,0,2024
4,COD,Agriculture,0,2024
...,...,...,...,...
427,SYR,Water Sanitation Hygiene,"13,563,289",2024
428,TCD,Water Sanitation Hygiene,"3,363,508",2024
429,UKR,Water Sanitation Hygiene,"9,560,978",2024
430,VEN,Water Sanitation Hygiene,"3,387,506",2024
